In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/shubhamyadav74/best-model-xgboost-18/scikitlearn/default/1/xgbr_0.180083.pkl
/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/best_time_recommendations.csv
/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/final_data.csv
/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/smoothing_tuning_metrics.csv
/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/__results__.html
/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/ui_ready_timeslot_output.csv
/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/driver_relocation_recommendations.csv
/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/region_slot_profile.csv
/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/historical_features.csv
/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/__notebook__.ipynb
/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/__output__.json
/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/custom.css


In [2]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
)

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

!pip install xgboost
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

!pip install mlflow
try:
    import mlflow
    HAS_MLFLOW = True
except Exception:
    HAS_MLFLOW = False

!pip install dagshub
try:
    import dagshub
    HAS_DAGSHUB = True
except Exception:
    HAS_DAGSHUB = False

import optuna

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.

In [3]:
# import dagshub
# import mlflow

# dagshub.init(
#     repo_owner="Shubham39275",
#     repo_name="Taxi_Updated",
#     mlflow=True
# )

# mlflow.set_experiment("Model Selection")

In [4]:
from pathlib import Path
import pandas as pd

# helper
def get_time_col(df):
    for c in ["pickup_slot", "tpep_pickup_datetime", "timestamp"]:
        if c in df.columns:
            return c
    return None

# Kaggle-aware paths
KAGGLE_INPUT = Path("/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522")
data_processed = Path("/kaggle/working")
data_interim = Path("/kaggle/working")

train_path = data_processed / "train.csv"
test_path = data_processed / "test.csv"

time_col = None

if train_path.exists() and test_path.exists():
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    source_mode = "processed_train_test"

    time_col = get_time_col(train_df) or get_time_col(test_df)
    if time_col:
        train_df[time_col] = pd.to_datetime(train_df[time_col], errors="coerce")
        test_df[time_col] = pd.to_datetime(test_df[time_col], errors="coerce")

else:
    hist_candidates = [
        data_interim / "historical_features.csv",
        data_interim / "final_data.csv",
        KAGGLE_INPUT / "historical_features.csv",
        KAGGLE_INPUT / "final_data.csv",
    ]

    hist_path = next((p for p in hist_candidates if p.exists()), None)

    if hist_path is None:
        raise FileNotFoundError(
            "No train/test or historical features found. Check Kaggle input path."
        )

    print("Using file:", hist_path)
    all_df = pd.read_csv(hist_path)

    time_col = get_time_col(all_df)
    if time_col is None:
        raise ValueError(
            "Historical data missing time column (`pickup_slot` or `tpep_pickup_datetime`)."
        )

    all_df[time_col] = pd.to_datetime(all_df[time_col], errors="coerce")
    all_df = all_df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)

    split_idx = int(len(all_df) * 0.8)
    train_df = all_df.iloc[:split_idx].copy()
    test_df = all_df.iloc[split_idx:].copy()
    source_mode = "historical_split"

    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

print("Source mode:", source_mode)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

if time_col and time_col in train_df.columns and time_col in test_df.columns:
    print("Train time range:", train_df[time_col].min(), "→", train_df[time_col].max())
    print("Test time range:", test_df[time_col].min(), "→", test_df[time_col].max())
else:
    print("No time column found in train/test for range debug.")


Using file: /kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/historical_features.csv
Source mode: historical_split
Train shape: (209664, 49)
Test shape: (52416, 49)
Train time range: 2016-01-01 00:00:00 → 2016-03-13 19:00:00
Test time range: 2016-03-13 19:00:00 → 2016-03-31 23:45:00


In [5]:
# def smape(y_true, y_pred):
#     y_true = np.asarray(y_true)
#     y_pred = np.asarray(y_pred)
#     denom = np.abs(y_true) + np.abs(y_pred)
#     return np.mean(2.0 * np.abs(y_true - y_pred) / np.where(denom == 0, 1.0, denom))


# def evaluate_metrics(y_true, y_pred):
#     return {
#         "MAE": float(mean_absolute_error(y_true, y_pred)),
#         "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
#         "MAPE": float(mean_absolute_percentage_error(y_true, y_pred)),
#         "sMAPE": float(smape(y_true, y_pred)),
#         "R2": float(r2_score(y_true, y_pred)),
#     }


# def safe_fill_frame(df: pd.DataFrame):
#     out = df.copy()
#     for col in out.columns:
#         if pd.api.types.is_numeric_dtype(out[col]):
#             median_val = out[col].median()
#             if pd.isna(median_val):
#                 median_val = 0.0
#             out[col] = out[col].fillna(median_val)
#         else:
#             mode_vals = out[col].mode(dropna=True)
#             fill_val = mode_vals.iloc[0] if len(mode_vals) else "unknown"
#             out[col] = out[col].fillna(fill_val)
#     return out


# def get_time_col(df: pd.DataFrame):
#     for c in ["pickup_slot", "tpep_pickup_datetime", "timestamp"]:
#         if c in df.columns:
#             return c
#     return None


In [6]:
# def safe_mape(y_true, y_pred):
#     y_true = np.maximum(y_true, 1)   # 🔥 avoid explosion when near 0
#     return np.mean(np.abs((y_true - y_pred) / y_true))


def safe_mape(y_true, y_pred):
    y_true = np.maximum(y_true, 10)   # 🔥 key trick
    return mean_absolute_percentage_error(y_true, y_pred)

def smape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.abs(y_true) + np.abs(y_pred)
    return np.mean(2.0 * np.abs(y_true - y_pred) / np.where(denom == 0, 1.0, denom))

def evaluate_metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE": float(safe_mape(y_true, y_pred)),   # 🔥 use safe version
        "sMAPE": float(smape(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }
def safe_fill_frame(df: pd.DataFrame):
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            median_val = out[col].median()
            if pd.isna(median_val):
                median_val = 0.0
            out[col] = out[col].fillna(median_val)
        else:
            mode_vals = out[col].mode(dropna=True)
            fill_val = mode_vals.iloc[0] if len(mode_vals) else "unknown"
            out[col] = out[col].fillna(fill_val)
    return out


def get_time_col(df: pd.DataFrame):
    for c in ["pickup_slot", "tpep_pickup_datetime", "timestamp"]:
        if c in df.columns:
            return c
    return None


In [7]:
# Load training/test data
train_path = data_processed / "train.csv"
test_path = data_processed / "test.csv"

if train_path.exists() and test_path.exists():
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    source_mode = "processed_train_test"
else:
    hist_candidates = [
        data_interim / "historical_features.csv",
        data_interim / "final_data.csv",
    ]
    hist_path = next((p for p in hist_candidates if p.exists()), None)
    if hist_path is None:
        raise FileNotFoundError(
            "No train/test or historical features found. Run Notebook 4 first."
        )

    all_df = pd.read_csv(hist_path)
    time_col = get_time_col(all_df)
    if time_col is None:
        raise ValueError("Historical data missing time column (`pickup_slot` or `tpep_pickup_datetime`).")

    all_df[time_col] = pd.to_datetime(all_df[time_col], errors="coerce")
    all_df = all_df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)

    split_idx = int(len(all_df) * 0.8)
    train_df = all_df.iloc[:split_idx].copy()
    test_df = all_df.iloc[split_idx:].copy()
    source_mode = "historical_split"

print("Source mode:", source_mode)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Source mode: processed_train_test
Train shape: (209664, 49)
Test shape: (52416, 49)


In [8]:
# # Target and feature preparation + lag feature engineering (FINAL FIX)

# target_candidates = ["total_pickups_model", "total_pickups_raw", "total_pickups"]
# target_col = next((c for c in target_candidates if c in train_df.columns), None)

# if target_col is None:
#     raise ValueError(f"Target column missing. Expected one of: {target_candidates}")

# time_col = get_time_col(train_df)
# if time_col is None:
#     raise ValueError("Time column required")

# # Ensure datetime
# train_df[time_col] = pd.to_datetime(train_df[time_col], errors="coerce")
# test_df[time_col] = pd.to_datetime(test_df[time_col], errors="coerce")

# # Sort
# sort_cols = ["region", time_col] if "region" in train_df.columns else [time_col]
# train_df = train_df.sort_values(sort_cols).reset_index(drop=True)
# test_df = test_df.sort_values(sort_cols).reset_index(drop=True)

# # ---------------- LAG FEATURES ----------------
# lag_steps = [1, 2, 3, 4, 6, 12, 24, 96]
# rolling_windows = [3, 6, 12]

# import numpy as np

# if "pickup_hour" in train_df.columns:
#     for df in [train_df, test_df]:
#         df["hour_sin"] = np.sin(2 * np.pi * df["pickup_hour"] / 24)
#         df["hour_cos"] = np.cos(2 * np.pi * df["pickup_hour"] / 24)

# def add_lag(df):
#     if "region" in df.columns:
#         for lag in lag_steps:
#             df[f"lag_{lag}"] = df.groupby("region")[target_col].shift(lag)

#         for w in rolling_windows:
#             df[f"lag_roll_mean_{w}"] = df.groupby("region")[target_col].transform(
#                 lambda s: s.shift(1).rolling(w).mean()
#             )
#             df[f"lag_roll_std_{w}"] = df.groupby("region")[target_col].transform(
#                 lambda s: s.shift(1).rolling(w).std()
#             )
#     else:
#         for lag in lag_steps:
#             df[f"lag_{lag}"] = df[target_col].shift(lag)

#         for w in rolling_windows:
#             df[f"lag_roll_mean_{w}"] = df[target_col].shift(1).rolling(w).mean()

#     return df

# # Train lag
# train_df = add_lag(train_df)
# lag_cols = [c for c in train_df.columns if c.startswith("lag_")]
# train_df = train_df.dropna(subset=lag_cols).reset_index(drop=True)

# # Test lag (only past)
# history = train_df.tail(150)
# test_temp = pd.concat([history, test_df], axis=0).reset_index(drop=True)
# test_temp = add_lag(test_temp)

# test_df = test_temp.iloc[len(history):].copy()
# test_df = test_df.dropna(subset=lag_cols).reset_index(drop=True)

# print("Train shape:", train_df.shape)
# print("Test shape:", test_df.shape)

# # ---------------- STRICT FEATURE SELECTION ----------------

# # ONLY allow SAFE columns
# safe_base_features = [
#     "region",
#     "pickup_hour",
#     "pickup_day_of_week",
#     "is_weekend",
#     "rush_hour",
#     "is_night",
#     "hour_sin",
#     "hour_cos",
# ]

# safe_base_features = [c for c in safe_base_features if c in train_df.columns]

# lag_features = [c for c in train_df.columns if c.startswith("lag_")]

# # FINAL FEATURES = ONLY THESE
# feature_cols = safe_base_features + lag_features

# if not feature_cols:
#     raise ValueError("No valid features")

# # ---------------- SPLIT ----------------

# X_train = train_df[feature_cols].copy()
# y_train = train_df[target_col].copy()

# X_test = test_df[feature_cols].copy()
# y_test = test_df[target_col].copy()

# X_train = safe_fill_frame(X_train)
# X_test = safe_fill_frame(X_test)

# # Convert boolean columns to int (fix sklearn error)
# for col in X_train.columns:
#     if X_train[col].dtype == "bool":
#         X_train[col] = X_train[col].astype(int)
#         X_test[col] = X_test[col].astype(int)

# print("Final Features:", feature_cols)
# print("Feature count:", len(feature_cols))

In [9]:
# #######
# # Target and feature preparation + lag feature engineering (IMPROVED)

# target_candidates = ["total_pickups_model", "total_pickups_raw", "total_pickups"]
# target_col = next((c for c in target_candidates if c in train_df.columns), None)

# if target_col is None:
#     raise ValueError(f"Target column missing. Expected one of: {target_candidates}")

# time_col = get_time_col(train_df)
# if time_col is None:
#     raise ValueError("Time column required")

# # Ensure datetime
# train_df[time_col] = pd.to_datetime(train_df[time_col], errors="coerce")
# test_df[time_col] = pd.to_datetime(test_df[time_col], errors="coerce")

# # Sort
# sort_cols = ["region", time_col] if "region" in train_df.columns else [time_col]
# train_df = train_df.sort_values(sort_cols).reset_index(drop=True)
# test_df = test_df.sort_values(sort_cols).reset_index(drop=True)

# # ---------------- LAG FEATURES ----------------
# lag_steps = [1, 2, 3, 4, 6, 12, 24, 96]
# rolling_windows = [3, 6, 12]

# import numpy as np

# # Cyclic time features
# if "pickup_hour" in train_df.columns:
#     for df in [train_df, test_df]:
#         df["hour_sin"] = np.sin(2 * np.pi * df["pickup_hour"] / 24)
#         df["hour_cos"] = np.cos(2 * np.pi * df["pickup_hour"] / 24)

# def add_lag(df):
#     if "region" in df.columns:
#         for lag in lag_steps:
#             df[f"lag_{lag}"] = df.groupby("region")[target_col].shift(lag)

#         for w in rolling_windows:
#             df[f"lag_roll_mean_{w}"] = df.groupby("region")[target_col].transform(
#                 lambda s: s.shift(1).rolling(w).mean()
#             )
#             df[f"lag_roll_std_{w}"] = df.groupby("region")[target_col].transform(
#                 lambda s: s.shift(1).rolling(w).std()
#             )
#     else:
#         for lag in lag_steps:
#             df[f"lag_{lag}"] = df[target_col].shift(lag)

#         for w in rolling_windows:
#             df[f"lag_roll_mean_{w}"] = df[target_col].shift(1).rolling(w).mean()

#     return df

# # Train lag
# train_df = add_lag(train_df)
# lag_cols = [c for c in train_df.columns if c.startswith("lag_")]
# train_df = train_df.dropna(subset=lag_cols).reset_index(drop=True)

# # Test lag (only past)
# history = train_df.tail(150)
# test_temp = pd.concat([history, test_df], axis=0).reset_index(drop=True)
# test_temp = add_lag(test_temp)

# test_df = test_temp.iloc[len(history):].copy()
# test_df = test_df.dropna(subset=lag_cols).reset_index(drop=True)

# # ---------------- NEW FEATURES (IMPORTANT) ----------------

# # Trend features
# train_df["lag_diff_1"] = train_df["lag_1"] - train_df["lag_2"]
# test_df["lag_diff_1"] = test_df["lag_1"] - test_df["lag_2"]

# train_df["lag_diff_24"] = train_df["lag_1"] - train_df["lag_24"]
# test_df["lag_diff_24"] = test_df["lag_1"] - test_df["lag_24"]

# train_df["trend_strength"] = train_df["lag_1"] - train_df["lag_roll_mean_3"]
# test_df["trend_strength"] = test_df["lag_1"] - test_df["lag_roll_mean_3"]

# # Peak hour
# peak_hours = [8, 9, 18, 19]
# train_df["is_peak"] = train_df["pickup_hour"].isin(peak_hours).astype(int)
# test_df["is_peak"] = test_df["pickup_hour"].isin(peak_hours).astype(int)

# # Region interaction
# train_df["region_hour"] = train_df["region"] * train_df["pickup_hour"]
# test_df["region_hour"] = test_df["region"] * test_df["pickup_hour"]

# print("Train shape:", train_df.shape)
# print("Test shape:", test_df.shape)

# # ---------------- FEATURE SELECTION ----------------

# safe_base_features = [
#     "region",
#     "pickup_hour",
#     "pickup_day_of_week",
#     "is_weekend",
#     "rush_hour",
#     "is_night",
#     "hour_sin",
#     "hour_cos",
# ]

# safe_base_features = [c for c in safe_base_features if c in train_df.columns]

# lag_features = [c for c in train_df.columns if c.startswith("lag_")]

# # feature_cols = safe_base_features + lag_features + [
# #     "lag_diff_1",
# #     "lag_diff_24",
# #     "trend_strength",
# #     "is_peak",
# #     "region_hour"
# # ]
# feature_cols = list(set(
#     safe_base_features + lag_features + [
#         "trend_strength",
#         "is_peak",
#         "region_hour"
#     ]
# ))

# X_train = train_df[feature_cols].copy()
# y_train = train_df[target_col].copy()

# X_test = test_df[feature_cols].copy()
# y_test = test_df[target_col].copy()

# X_train = safe_fill_frame(X_train)
# X_test = safe_fill_frame(X_test)

# # Fix bool issue
# bool_cols = X_train.select_dtypes(include=["bool"]).columns

# for col in bool_cols:
#     X_train[col] = X_train[col].astype(int)
#     X_test[col] = X_test[col].astype(int)

# print("Final Features:", feature_cols)
# print("Feature count:", len(feature_cols))

In [10]:
# # ================= FINAL FEATURE ENGINEERING =================

# target_candidates = ["total_pickups_model", "total_pickups_raw", "total_pickups"]
# target_col = next((c for c in target_candidates if c in train_df.columns), None)

# if target_col is None:
#     raise ValueError(f"Target column missing. Expected one of: {target_candidates}")

# time_col = get_time_col(train_df)
# if time_col is None:
#     raise ValueError("Time column required")

# # Ensure datetime
# train_df[time_col] = pd.to_datetime(train_df[time_col], errors="coerce")
# test_df[time_col] = pd.to_datetime(test_df[time_col], errors="coerce")

# # Sort
# sort_cols = ["region", time_col] if "region" in train_df.columns else [time_col]
# train_df = train_df.sort_values(sort_cols).reset_index(drop=True)
# test_df = test_df.sort_values(sort_cols).reset_index(drop=True)

# # ---------------- LAG FEATURES ----------------
# lag_steps = [1, 2, 3, 6, 12, 24]   # 🔥 reduced noise
# rolling_windows = [3, 6]

# import numpy as np

# # Cyclic features
# if "pickup_hour" in train_df.columns:
#     for df in [train_df, test_df]:
#         df["hour_sin"] = np.sin(2 * np.pi * df["pickup_hour"] / 24)
#         df["hour_cos"] = np.cos(2 * np.pi * df["pickup_hour"] / 24)

# def add_lag(df):
#     if "region" in df.columns:
#         for lag in lag_steps:
#             df[f"lag_{lag}"] = df.groupby("region")[target_col].shift(lag)

#         for w in rolling_windows:
#             df[f"lag_roll_mean_{w}"] = df.groupby("region")[target_col].transform(
#                 lambda s: s.shift(1).rolling(w).mean()
#             )
#             df[f"lag_roll_std_{w}"] = df.groupby("region")[target_col].transform(
#                 lambda s: s.shift(1).rolling(w).std()
#             )
#     else:
#         for lag in lag_steps:
#             df[f"lag_{lag}"] = df[target_col].shift(lag)

#     return df

# # Apply lag
# train_df = add_lag(train_df)
# lag_cols = [c for c in train_df.columns if c.startswith("lag_")]
# train_df = train_df.dropna(subset=lag_cols).reset_index(drop=True)

# # Test lag using history
# history = train_df.tail(150)
# test_temp = pd.concat([history, test_df], axis=0).reset_index(drop=True)
# test_temp = add_lag(test_temp)

# test_df = test_temp.iloc[len(history):].copy()
# test_df = test_df.dropna(subset=lag_cols).reset_index(drop=True)

# # ---------------- NEW POWER FEATURES ----------------

# # Trend
# train_df["trend_strength"] = train_df["lag_1"] - train_df["lag_roll_mean_3"]
# test_df["trend_strength"] = test_df["lag_1"] - test_df["lag_roll_mean_3"]

# # Peak
# peak_hours = [8, 9, 18, 19]
# train_df["is_peak"] = train_df["pickup_hour"].isin(peak_hours).astype(int)
# test_df["is_peak"] = test_df["pickup_hour"].isin(peak_hours).astype(int)

# # Region interaction
# train_df["region_hour"] = train_df["region"] * train_df["pickup_hour"]
# test_df["region_hour"] = test_df["region"] * test_df["pickup_hour"]

# # 🔥 MOST IMPORTANT (STAT FEATURES)
# # train_df["region_mean"] = train_df.groupby("region")[target_col].transform("mean")
# # test_df["region_mean"] = test_df.groupby("region")[target_col].transform("mean")

# # train_df["hour_mean"] = train_df.groupby("pickup_hour")[target_col].transform("mean")
# # test_df["hour_mean"] = test_df.groupby("pickup_hour")[target_col].transform("mean")

# # train_df["dow_mean"] = train_df.groupby("pickup_day_of_week")[target_col].transform("mean")
# # test_df["dow_mean"] = test_df.groupby("pickup_day_of_week")[target_col].transform("mean")

# # 🔥 FIXED (NO LEAKAGE)

# train_df["region_mean"] = train_df.groupby("region")[target_col].transform(
#     lambda s: s.shift(1).expanding().mean()
# )
# test_df["region_mean"] = test_df.groupby("region")[target_col].transform("mean")

# train_df["hour_mean"] = train_df.groupby("pickup_hour")[target_col].transform(
#     lambda s: s.shift(1).expanding().mean()
# )
# test_df["hour_mean"] = test_df.groupby("pickup_hour")[target_col].transform("mean")

# train_df["dow_mean"] = train_df.groupby("pickup_day_of_week")[target_col].transform(
#     lambda s: s.shift(1).expanding().mean()
# )
# test_df["dow_mean"] = test_df.groupby("pickup_day_of_week")[target_col].transform("mean")

# print("Train shape:", train_df.shape)
# print("Test shape:", test_df.shape)

# # ---------------- FEATURE SELECTION ----------------

# safe_base_features = [
#     "region",
#     "pickup_hour",
#     "pickup_day_of_week",
#     "is_weekend",
#     "rush_hour",
#     "is_night",
#     "hour_sin",
#     "hour_cos",
# ]

# safe_base_features = [c for c in safe_base_features if c in train_df.columns]

# lag_features = [c for c in train_df.columns if c.startswith("lag_")]

# extra_features = [
#     "trend_strength",
#     "is_peak",
#     "region_hour",
#     "region_mean",
#     "hour_mean",
#     "dow_mean",
# ]

# feature_cols = safe_base_features + lag_features + extra_features

# # remove duplicates safely
# feature_cols = list(dict.fromkeys(feature_cols))

# X_train = train_df[feature_cols].copy()
# y_train = train_df[target_col].copy()

# X_test = test_df[feature_cols].copy()
# y_test = test_df[target_col].copy()

# X_train = safe_fill_frame(X_train)
# X_test = safe_fill_frame(X_test)

# # Fix bool columns
# bool_cols = X_train.select_dtypes(include=["bool"]).columns
# for col in bool_cols:
#     X_train[col] = X_train[col].astype(int)
#     X_test[col] = X_test[col].astype(int)

# print("Final Features:", feature_cols)
# print("Feature count:", len(feature_cols))

In [11]:
# ================= FINAL FEATURE ENGINEERING =================

target_candidates = ["total_pickups_model", "total_pickups_raw", "total_pickups"]
target_col = next((c for c in target_candidates if c in train_df.columns), None)

if target_col is None:
    raise ValueError(f"Target column missing. Expected one of: {target_candidates}")

time_col = get_time_col(train_df)
if time_col is None:
    raise ValueError("Time column required")

# Ensure datetime
train_df[time_col] = pd.to_datetime(train_df[time_col], errors="coerce")
test_df[time_col] = pd.to_datetime(test_df[time_col], errors="coerce")

# Sort
sort_cols = ["region", time_col] if "region" in train_df.columns else [time_col]
train_df = train_df.sort_values(sort_cols).reset_index(drop=True)
test_df = test_df.sort_values(sort_cols).reset_index(drop=True)

# ================= TIME FEATURES =================
for df in [train_df, test_df]:
    df["week_of_year"] = df[time_col].dt.isocalendar().week.astype(int)
    df["day_of_month"] = df[time_col].dt.day
    df["is_month_start"] = df[time_col].dt.is_month_start.astype(int)
    df["is_month_end"] = df[time_col].dt.is_month_end.astype(int)

# ================= LAG FEATURES =================
lag_steps = [1, 2, 3, 6, 12, 24]
rolling_windows = [3, 6]

# Cyclic features
if "pickup_hour" in train_df.columns:
    for df in [train_df, test_df]:
        df["hour_sin"] = np.sin(2 * np.pi * df["pickup_hour"] / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df["pickup_hour"] / 24)


def add_lag(df):
    if "region" in df.columns:
        for lag in lag_steps:
            df[f"lag_{lag}"] = df.groupby("region")[target_col].shift(lag)

        for w in rolling_windows:
            df[f"lag_roll_mean_{w}"] = df.groupby("region")[target_col].transform(
                lambda s: s.shift(1).rolling(w).mean()
            )
            df[f"lag_roll_std_{w}"] = df.groupby("region")[target_col].transform(
                lambda s: s.shift(1).rolling(w).std()
            )
    else:
        for lag in lag_steps:
            df[f"lag_{lag}"] = df[target_col].shift(lag)

    return df


# train_df = add_lag(train_df)
# lag_cols = [c for c in train_df.columns if c.startswith("lag_")]
# train_df = train_df.dropna(subset=lag_cols).reset_index(drop=True)

train_df = add_lag(train_df)

lag_cols = [c for c in train_df.columns if c.startswith("lag_")]

# 🔥 SAFE DROP (CRITICAL FIX)
min_required_lags = [c for c in lag_cols if "lag_1" in c or "lag_2" in c]

train_df = train_df.dropna(subset=min_required_lags)

# fallback if still empty
if len(train_df) == 0:
    print("⚠️ train empty after drop → using forward fill fallback")
    train_df = add_lag(train_df)
    train_df = train_df.fillna(method="ffill").fillna(method="bfill")

train_df = train_df.reset_index(drop=True)

# test lag
history = train_df.tail(150)
test_temp = pd.concat([history, test_df], axis=0).reset_index(drop=True)
test_temp = add_lag(test_temp)

# test_df = test_temp.iloc[len(history):].copy()
# test_df = test_df.dropna(subset=lag_cols).reset_index(drop=True)

test_df = test_temp.iloc[len(history):].copy()

# 🔥 SAFE DROP
test_df = test_df.dropna(subset=min_required_lags)

if len(test_df) == 0:
    print("⚠️ test empty → fallback fill")
    test_df = test_df.fillna(method="ffill").fillna(method="bfill")

test_df = test_df.reset_index(drop=True)

# ================= EXTRA FEATURES =================

train_df["trend_strength"] = train_df["lag_1"] - train_df["lag_roll_mean_3"]
test_df["trend_strength"] = test_df["lag_1"] - test_df["lag_roll_mean_3"]

peak_hours = [8, 9, 18, 19]
train_df["is_peak"] = train_df["pickup_hour"].isin(peak_hours).astype(int)
test_df["is_peak"] = test_df["pickup_hour"].isin(peak_hours).astype(int)

train_df["region_hour"] = train_df["region"] * train_df["pickup_hour"]
test_df["region_hour"] = test_df["region"] * test_df["pickup_hour"]

# ================= REGION SMOOTHING (🔥 BEST FIX) =================

global_mean = train_df[target_col].mean()

region_stats = train_df.groupby("region")[target_col].agg(["mean", "count"])

smooth = 20

region_smooth_map = (
    (region_stats["mean"] * region_stats["count"] + global_mean * smooth)
    / (region_stats["count"] + smooth)
)

train_df["region_mean"] = train_df["region"].map(region_smooth_map)
test_df["region_mean"] = test_df["region"].map(region_smooth_map)

# other stats (safe version)
train_df["hour_mean"] = train_df.groupby("pickup_hour")[target_col].transform(
    lambda s: s.shift(1).expanding().mean()
)
test_df["hour_mean"] = test_df.groupby("pickup_hour")[target_col].transform("mean")

train_df["dow_mean"] = train_df.groupby("pickup_day_of_week")[target_col].transform(
    lambda s: s.shift(1).expanding().mean()
)
test_df["dow_mean"] = test_df.groupby("pickup_day_of_week")[target_col].transform("mean")

# ================= FEATURES =================

safe_base_features = [
    "region",
    "pickup_hour",
    "pickup_day_of_week",
    "is_weekend",
    "rush_hour",
    "is_night",
    "hour_sin",
    "hour_cos",
    "week_of_year",
    "day_of_month",
    "is_month_start",
    "is_month_end",
]

safe_base_features = [c for c in safe_base_features if c in train_df.columns]

lag_features = [c for c in train_df.columns if c.startswith("lag_")]

extra_features = [
    "trend_strength",
    "is_peak",
    "region_hour",
    "region_mean",
    "hour_mean",
    "dow_mean",
]

feature_cols = list(dict.fromkeys(safe_base_features + lag_features + extra_features))

X_train = safe_fill_frame(train_df[feature_cols])
y_train = train_df[target_col]

X_test = safe_fill_frame(test_df[feature_cols])
y_test = test_df[target_col]

# bool fix
for col in X_train.select_dtypes(include=["bool"]).columns:
    X_train[col] = X_train[col].astype(int)
    X_test[col] = X_test[col].astype(int)

print("Final Features:", feature_cols)
print("Feature count:", len(feature_cols))

Final Features: ['region', 'pickup_hour', 'pickup_day_of_week', 'is_weekend', 'rush_hour', 'is_night', 'hour_sin', 'hour_cos', 'week_of_year', 'day_of_month', 'is_month_start', 'is_month_end', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_24', 'lag_roll_mean_3', 'lag_roll_std_3', 'lag_roll_mean_6', 'lag_roll_std_6', 'trend_strength', 'is_peak', 'region_hour', 'region_mean', 'hour_mean', 'dow_mean']
Feature count: 28


In [12]:
# Time-aware validation split from training set
if time_col is not None and time_col in train_df.columns:
    train_sorted_idx = train_df.sort_values(time_col).index
    X_train_sorted = X_train.loc[train_sorted_idx]
    y_train_sorted = y_train.loc[train_sorted_idx]
else:
    X_train_sorted = X_train.copy()
    y_train_sorted = y_train.copy()

split_idx = int(len(X_train_sorted) * 0.8)
X_fit = X_train_sorted.iloc[:split_idx].copy()
y_fit = y_train_sorted.iloc[:split_idx].copy()

X_valid = X_train_sorted.iloc[split_idx:].copy()
y_valid = y_train_sorted.iloc[split_idx:].copy()

print("Fit split:", X_fit.shape, "| Validation split:", X_valid.shape)


Fit split: (167155, 28) | Validation split: (41789, 28)


In [13]:
# Detect categorical and numerical columns (SAFE)

cat_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

num_cols = X_train.select_dtypes(
    exclude=["object", "category", "bool"]
).columns.tolist()

print("Categorical:", cat_cols)
print("Numerical:", num_cols)

Categorical: []
Numerical: ['region', 'pickup_hour', 'pickup_day_of_week', 'is_weekend', 'rush_hour', 'is_night', 'hour_sin', 'hour_cos', 'week_of_year', 'day_of_month', 'is_month_start', 'is_month_end', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_24', 'lag_roll_mean_3', 'lag_roll_std_3', 'lag_roll_mean_6', 'lag_roll_std_6', 'trend_strength', 'is_peak', 'region_hour', 'region_mean', 'hour_mean', 'dow_mean']


In [14]:
# # NaN-safe preprocessing pipelines
# numeric_pipeline = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
# ])

# categorical_pipeline = Pipeline([
#     ("imputer", SimpleImputer(strategy="most_frequent")),
#     ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
# ])

# transformers = []
# if cat_cols:
#     transformers.append(("cat", categorical_pipeline, cat_cols))
# if num_cols:
#     transformers.append(("num", numeric_pipeline, num_cols))

# preprocessor = ColumnTransformer(
#     transformers=transformers,
#     remainder="drop",
# )


# def make_model_from_trial(trial):
#     options = ["LR", "RIDGE", "RF", "GBR"]
#     if HAS_XGB:
#         options.append("XGBR")

#     model_name = trial.suggest_categorical("model_name", options)

#     if model_name == "LR":
#         model = LinearRegression()

#     elif model_name == "RIDGE":
#         alpha = trial.suggest_float("ridge_alpha", 0.1, 200.0, log=True)
#         model = Ridge(alpha=alpha, random_state=RANDOM_STATE)

#     elif model_name == "RF":
#         n_estimators = trial.suggest_int("rf_n_estimators", 100, 400, step=50)
#         max_depth = trial.suggest_int("rf_max_depth", 5, 24)
#         min_samples_leaf = trial.suggest_int("rf_min_samples_leaf", 1, 8)
#         model = RandomForestRegressor(
#             n_estimators=n_estimators,
#             max_depth=max_depth,
#             min_samples_leaf=min_samples_leaf,
#             random_state=RANDOM_STATE,
#             n_jobs=-1,
#         )

#     elif model_name == "GBR":
#         n_estimators = trial.suggest_int("gbr_n_estimators", 80, 400, step=40)
#         learning_rate = trial.suggest_float("gbr_learning_rate", 0.01, 0.2, log=True)
#         max_depth = trial.suggest_int("gbr_max_depth", 2, 8)
#         subsample = trial.suggest_float("gbr_subsample", 0.6, 1.0)
#         model = GradientBoostingRegressor(
#             n_estimators=n_estimators,
#             learning_rate=learning_rate,
#             max_depth=max_depth,
#             subsample=subsample,
#             random_state=RANDOM_STATE,
#         )

#     else:  # XGBR
#         n_estimators = trial.suggest_int("xgb_n_estimators", 200, 800, step=100)
#         learning_rate = trial.suggest_float("xgb_learning_rate", 0.03, 0.2, log=True)
#         max_depth = trial.suggest_int("xgb_max_depth", 3, 8)
#         subsample = trial.suggest_float("xgb_subsample", 0.7, 1.0)
#         colsample_bytree = trial.suggest_float("xgb_colsample", 0.7, 1.0)
        
#         reg_alpha = trial.suggest_float("xgb_alpha", 0.0, 5.0)
#         reg_lambda = trial.suggest_float("xgb_lambda", 0.5, 5.0)
        
#         model = XGBRegressor(
#             n_estimators=n_estimators,
#             learning_rate=learning_rate,
#             max_depth=max_depth,
#             subsample=subsample,
#             colsample_bytree=colsample_bytree,
#             reg_alpha=reg_alpha,
#             reg_lambda=reg_lambda,
#             objective="reg:squarederror",
#             random_state=RANDOM_STATE,
#             n_jobs=-1,
#         )
#     return model_name, model


# def objective(trial):
#     model_name, model = make_model_from_trial(trial)

#     pipeline = Pipeline([
#         ("prep", preprocessor),
#         ("model", model),
#     ])

#     if HAS_MLFLOW:
#         mlflow.start_run(nested=True)
#         mlflow.log_param("model_name", model_name)

#     pipeline.fit(X_fit, y_fit)
#     y_pred_valid = pipeline.predict(X_valid)

#     metrics = evaluate_metrics(y_valid, y_pred_valid)

#     if HAS_MLFLOW:
#         for k, v in metrics.items():
#             mlflow.log_metric(f"valid_{k}", v)
#         mlflow.log_params(model.get_params())
#         mlflow.end_run()

#     return metrics["MAPE"]


In [15]:
# #####
# # NaN-safe preprocessing pipelines
# numeric_pipeline = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
# ])

# categorical_pipeline = Pipeline([
#     ("imputer", SimpleImputer(strategy="most_frequent")),
#     ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
# ])

# transformers = []
# if cat_cols:
#     transformers.append(("cat", categorical_pipeline, cat_cols))
# if num_cols:
#     transformers.append(("num", numeric_pipeline, num_cols))

# preprocessor = ColumnTransformer(
#     transformers=transformers,
#     remainder="drop",
# )

# def make_model_from_trial(trial):
#     model_name = trial.suggest_categorical(
#         "model_name",
#         ["LR", "RIDGE", "RF", "GBR", "XGBR"]
#     )

#     if model_name == "LR":
#         model = LinearRegression()

#     elif model_name == "RIDGE":
#         alpha = trial.suggest_float("ridge_alpha", 0.1, 100.0, log=True)
#         model = Ridge(alpha=alpha, random_state=RANDOM_STATE)

#     elif model_name == "RF":
#         model = RandomForestRegressor(
#             n_estimators=trial.suggest_int("rf_n_estimators", 50, 200, step=50),
#             max_depth=trial.suggest_int("rf_max_depth", 5, 15),
#             min_samples_leaf=trial.suggest_int("rf_min_samples_leaf", 2, 10),
#             max_features=trial.suggest_float("rf_max_features", 0.5, 1.0),
#             random_state=RANDOM_STATE,
#             n_jobs=-1,
#         )

#     elif model_name == "GBR":
#         model = GradientBoostingRegressor(
#             n_estimators=trial.suggest_int("gbr_n_estimators", 80, 200, step=40),
#             learning_rate=trial.suggest_float("gbr_learning_rate", 0.01, 0.1, log=True),
#             max_depth=trial.suggest_int("gbr_max_depth", 2, 6),
#             subsample=trial.suggest_float("gbr_subsample", 0.7, 1.0),
#             random_state=RANDOM_STATE,
#         )

#     else:  # XGBR
#         model = XGBRegressor(
#             n_estimators=trial.suggest_int("xgb_n_estimators", 200, 600, step=100),
#             learning_rate=trial.suggest_float("xgb_learning_rate", 0.03, 0.15, log=True),
#             max_depth=trial.suggest_int("xgb_max_depth", 3, 8),
#             subsample=trial.suggest_float("xgb_subsample", 0.7, 1.0),
#             colsample_bytree=trial.suggest_float("xgb_colsample", 0.7, 1.0),
#             reg_alpha=trial.suggest_float("xgb_alpha", 0.0, 3.0),
#             reg_lambda=trial.suggest_float("xgb_lambda", 0.5, 3.0),
#             tree_method="hist",
#             objective="reg:squarederror",
#             random_state=RANDOM_STATE,
#             n_jobs=-1,
#         )

#     return model_name, model


# def objective(trial):
#     model_name, model = make_model_from_trial(trial)

#     pipeline = Pipeline([
#         ("prep", preprocessor),
#         ("model", model),
#     ])

#     # 🔥 Speed optimization (IMPORTANT)
#     X_fit_sample = X_fit.sample(n=50000, random_state=42)
#     y_fit_sample = y_fit.loc[X_fit_sample.index]

#     if HAS_MLFLOW:
#         mlflow.start_run(nested=True)
#         mlflow.log_param("model_name", model_name)

#     pipeline.fit(X_fit_sample, y_fit_sample)
#     y_pred_valid = pipeline.predict(X_valid)

#     metrics = evaluate_metrics(y_valid, y_pred_valid)

#     if HAS_MLFLOW:
#         for k, v in metrics.items():
#             mlflow.log_metric(f"valid_{k}", v)
#         mlflow.log_params(model.get_params())
#         mlflow.end_run()

#     return metrics["MAPE"]

In [16]:
# ================= OPTUNA PIPELINE (FIXED) =================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

transformers = []
if cat_cols:
    transformers.append(("cat", categorical_pipeline, cat_cols))
if num_cols:
    transformers.append(("num", numeric_pipeline, num_cols))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

def make_model_from_trial(trial):
    # 🔥 biased toward strong models
    model_name = trial.suggest_categorical(
        "model_name",
        ["XGBR", "XGBR", "RF", "RF", "GBR"]
    )

    if model_name == "RF":
        model = RandomForestRegressor(
            n_estimators=trial.suggest_int("rf_n_estimators", 100, 200, step=50),
            max_depth=trial.suggest_int("rf_max_depth", 8, 18),
            min_samples_leaf=trial.suggest_int("rf_min_samples_leaf", 2, 8),
            max_features=trial.suggest_float("rf_max_features", 0.5, 0.9),
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    elif model_name == "GBR":
        model = GradientBoostingRegressor(
            n_estimators=trial.suggest_int("gbr_n_estimators", 80, 160, step=40),
            learning_rate=trial.suggest_float("gbr_learning_rate", 0.01, 0.1, log=True),
            max_depth=trial.suggest_int("gbr_max_depth", 2, 5),
            subsample=trial.suggest_float("gbr_subsample", 0.7, 1.0),
            random_state=RANDOM_STATE,
        )

    else:  # XGBR
        model = XGBRegressor(
            n_estimators=trial.suggest_int("xgb_n_estimators", 200, 500, step=100),
            learning_rate=trial.suggest_float("xgb_learning_rate", 0.03, 0.15, log=True),
            max_depth=trial.suggest_int("xgb_max_depth", 3, 7),
            subsample=trial.suggest_float("xgb_subsample", 0.7, 1.0),
            colsample_bytree=trial.suggest_float("xgb_colsample", 0.7, 1.0),
            reg_alpha=trial.suggest_float("xgb_alpha", 0.0, 3.0),
            reg_lambda=trial.suggest_float("xgb_lambda", 0.5, 3.0),
            tree_method="hist",
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    return model_name, model


# def objective(trial):
#     model_name, model = make_model_from_trial(trial)

#     pipeline = Pipeline([
#         ("prep", preprocessor),
#         ("model", model),
#     ])

#     # 🔥 faster but stable
#     sample_size = min(50000, len(X_fit))
#     X_fit_sample = X_fit.sample(n=sample_size, random_state=42)
#     y_fit_sample = y_fit.loc[X_fit_sample.index]

#     pipeline.fit(X_fit_sample, y_fit_sample)
#     y_pred_valid = pipeline.predict(X_valid)

#     metrics = evaluate_metrics(y_valid, y_pred_valid)

#     return metrics["MAPE"]

# def objective(trial):
#     model_name, model = make_model_from_trial(trial)

#     pipeline = Pipeline([
#         ("prep", preprocessor),
#         ("model", model),
#     ])

#     sample_size = min(50000, len(X_fit))
#     X_fit_sample = X_fit.sample(n=sample_size, random_state=42)
#     y_fit_sample = y_fit.loc[X_fit_sample.index]

#     # ----------------------------✅ CHANGE 1: log transform
#     # y_log = np.log1p(y_fit_sample)

#     # try sqrt (EXPERIMENT)
#     y_log = np.sqrt(y_fit_sample)

#     pipeline.fit(X_fit_sample, y_log)

#     # ✅ CHANGE 2: reverse transform
#     y_pred_log = pipeline.predict(X_valid)
#     # y_pred = np.expm1(y_pred_log)
    
#     # ------------------------------------- try sqrt (EXPERIMENT)
#     y_pred = y_pred_log ** 2

#     # ✅ CHANGE 3: stable MAPE
#     return safe_mape(y_valid, y_pred)

def objective(trial):
    model_name, model = make_model_from_trial(trial)

    pipeline = Pipeline([
        ("prep", preprocessor),
        ("model", model),
    ])

    sample_size = min(50000, len(X_fit))
    X_fit_sample = X_fit.sample(n=sample_size, random_state=42)
    y_fit_sample = y_fit.loc[X_fit_sample.index]

    # 🔥 LOG TRANSFORM
    y_log = np.log1p(y_fit_sample)

    pipeline.fit(X_fit_sample, y_log)

    # predict
    y_pred_log = pipeline.predict(X_valid)
    y_pred = np.expm1(y_pred_log)

    return safe_mape(y_valid, y_pred)

In [17]:
# # Run Optuna model selection
# n_trials = 50

# study = optuna.create_study(
#     study_name="model_selection",
#     direction="minimize",
#     sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
# )

# if HAS_MLFLOW:
#     with mlflow.start_run(run_name="model_selection_optuna"):
#         study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True, catch=(ValueError,))
#         mlflow.log_params(study.best_params)
#         mlflow.log_metric("best_valid_MAPE", study.best_value)
# else:
#     study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True, catch=(ValueError,))

# print("Best validation MAPE:", round(study.best_value, 6))
# print("Best params:")
# study.best_params


In [18]:
# study.trials_dataframe().sort_values("value").head(10)

In [19]:
print("Final Features:", feature_cols)
print("Feature count:", len(feature_cols))

# 🔥 DEBUG CHECK (ADD THIS)
print("After FE:")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Final Features: ['region', 'pickup_hour', 'pickup_day_of_week', 'is_weekend', 'rush_hour', 'is_night', 'hour_sin', 'hour_cos', 'week_of_year', 'day_of_month', 'is_month_start', 'is_month_end', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_24', 'lag_roll_mean_3', 'lag_roll_std_3', 'lag_roll_mean_6', 'lag_roll_std_6', 'trend_strength', 'is_peak', 'region_hour', 'region_mean', 'hour_mean', 'dow_mean']
Feature count: 28
After FE:
Train shape: (208944, 71)
Test shape: (51720, 71)


In [20]:
# # ================= SAVE BEST MODEL =================

# best_model_name = study.best_params["model_name"]

# # rebuild best model
# _, best_model = make_model_from_trial(optuna.trial.FixedTrial(study.best_params))

# final_pipeline = Pipeline([
#     ("prep", preprocessor),
#     ("model", best_model),
# ])

# # train on FULL training data
# final_pipeline.fit(X_train, y_train)

# # save
# joblib.dump(final_pipeline, "/kaggle/working/final_xgbr_0.179127.pkl")

# print("✅ Model saved successfully!")

“After adding temporal and cyclic features, tree-based models—especially XGBoost with regularization—started outperforming simpler models, indicating complex non-linear demand patterns.”

In [21]:
import pandas as pd
import numpy as np

data_path = "/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/final_data.csv"
df = pd.read_csv(data_path)

def get_time_col(df):
    for c in ["pickup_slot", "tpep_pickup_datetime", "timestamp"]:
        if c in df.columns:
            return c
    return None

time_col = get_time_col(df)

df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
df = df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)

split_date = df[time_col].max() - pd.Timedelta(days=15)

train_df = df[df[time_col] < split_date].copy()
test_df = df[df[time_col] >= split_date].copy()

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (218850, 49)
Test: (43230, 49)


In [22]:
target_candidates = ["total_pickups_model", "total_pickups_raw", "total_pickups"]
target_col = next((c for c in target_candidates if c in train_df.columns), None)

# TIME FEATURES
for df_ in [train_df, test_df]:
    df_["week_of_year"] = df_[time_col].dt.isocalendar().week.astype(int)
    df_["day_of_month"] = df_[time_col].dt.day
    df_["is_month_start"] = df_[time_col].dt.is_month_start.astype(int)
    df_["is_month_end"] = df_[time_col].dt.is_month_end.astype(int)

# LAG FEATURES
lag_steps = [1,2,3,6,12,24]
rolling_windows = [3,6]

def add_lag(df):
    for lag in lag_steps:
        df[f"lag_{lag}"] = df.groupby("region")[target_col].shift(lag)

    for w in rolling_windows:
        df[f"lag_roll_mean_{w}"] = df.groupby("region")[target_col].transform(
            lambda s: s.shift(1).rolling(w).mean()
        )
        df[f"lag_roll_std_{w}"] = df.groupby("region")[target_col].transform(
            lambda s: s.shift(1).rolling(w).std()
        )
    return df

train_df = add_lag(train_df)
lag_cols = [c for c in train_df.columns if c.startswith("lag_")]

# 🔥 SAFE DROP (avoid empty train)
train_df = train_df.dropna(subset=lag_cols)
if len(train_df) == 0:
    print("⚠️ fallback to fill")
    train_df = train_df.fillna(method="ffill").fillna(method="bfill")

train_df = train_df.reset_index(drop=True)

# TEST FIX
history = train_df.tail(200)
test_temp = pd.concat([history, test_df]).reset_index(drop=True)
test_temp = add_lag(test_temp)

test_df = test_temp.iloc[len(history):].copy()
test_df = test_df.fillna(method="ffill").fillna(method="bfill")

if len(test_df) == 0:
    raise ValueError("❌ Test empty after FE")

test_df = test_df.reset_index(drop=True)

print("After FE:", train_df.shape, test_df.shape)

After FE: (218130, 63) (43230, 63)


In [23]:
train_df["trend_strength"] = train_df["lag_1"] - train_df["lag_roll_mean_3"]
test_df["trend_strength"] = test_df["lag_1"] - test_df["lag_roll_mean_3"]

train_df["region_hour"] = train_df["region"] * train_df["pickup_hour"]
test_df["region_hour"] = test_df["region"] * test_df["pickup_hour"]

global_mean = train_df[target_col].mean()
region_stats = train_df.groupby("region")[target_col].agg(["mean", "count"])

smooth = 20
region_map = (
    (region_stats["mean"] * region_stats["count"] + global_mean * smooth)
    / (region_stats["count"] + smooth)
)

train_df["region_mean"] = train_df["region"].map(region_map)
test_df["region_mean"] = test_df["region"].map(region_map)

In [24]:
feature_cols = [c for c in train_df.columns if c.startswith("lag_")] + [
    "region","pickup_hour","pickup_day_of_week",
    "week_of_year","day_of_month","is_month_start","is_month_end",
    "trend_strength","region_hour","region_mean"
]

# 🔥 SAFETY
feature_cols = [c for c in feature_cols if c in train_df.columns]

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (218130, 20)
X_test: (43230, 20)


In [25]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.1026,
    max_depth=6,
    subsample=0.8295,
    colsample_bytree=0.9024,
    reg_alpha=1.2257,
    reg_lambda=0.8597,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

y_train_log = np.log1p(y_train)
model.fit(X_train, y_train_log)

y_pred_train = np.expm1(model.predict(X_train))
y_pred_test = np.expm1(model.predict(X_test))

y_pred_train = np.clip(y_pred_train, 0, None)
y_pred_test = np.clip(y_pred_test, 0, None)

In [26]:
predictions = pd.DataFrame({
    "actual_demand": y_test.values,
    "predicted_demand": y_pred_test,
}).reset_index(drop=True)

context_test = test_df.iloc[:len(predictions)].copy()

if time_col in context_test.columns:
    predictions["pickup_slot"] = pd.to_datetime(context_test[time_col].values)

if "region" in context_test.columns:
    predictions["region"] = context_test["region"].values

predictions["rolling_mean"] = predictions.groupby("region")["actual_demand"].transform("mean")
predictions["rolling_mean"] = predictions["rolling_mean"].fillna(predictions["rolling_mean"].median())

predictions["surge_threshold"] = predictions["rolling_mean"] * 1.3
predictions["surge_flag"] = predictions["predicted_demand"] > predictions["surge_threshold"]

ratio = predictions["predicted_demand"] / (predictions["surge_threshold"] + 1e-6)

predictions["surge_level"] = "none"
predictions.loc[(predictions["surge_flag"]) & (ratio <= 1.1), "surge_level"] = "low"
predictions.loc[(predictions["surge_flag"]) & (ratio > 1.1) & (ratio <= 1.25), "surge_level"] = "medium"
predictions.loc[(predictions["surge_flag"]) & (ratio > 1.25), "surge_level"] = "high"

predictions["rolling_std"] = predictions.groupby("region")["actual_demand"].transform("std").fillna(0)
predictions["risk_score"] = predictions["rolling_std"] / (predictions["rolling_mean"] + 1e-6)

predictions["risk_band"] = pd.cut(
    predictions["risk_score"],
    bins=[-np.inf, 0.35, 0.75, np.inf],
    labels=["Stable", "Moderate", "Volatile"]
).astype(str)

if "avg_fare_region_slot" in test_df.columns:
    predictions["avg_fare"] = context_test["avg_fare_region_slot"].values
elif "avg_fare_region_slot" in train_df.columns:
    predictions["avg_fare"] = train_df["avg_fare_region_slot"].median()
else:
    predictions["avg_fare"] = 10

predictions["expected_revenue"] = predictions["predicted_demand"] * predictions["avg_fare"]

baseline = predictions.groupby("region")["actual_demand"].transform("mean")
predictions["demand_pressure"] = predictions["predicted_demand"] / (baseline + 1e-6)

if "pickup_slot" in predictions.columns:
    predictions["pickup_hour"] = predictions["pickup_slot"].dt.hour
    predictions["pickup_day"] = predictions["pickup_slot"].dt.dayofweek

best_times = (
    predictions.groupby(["region", "pickup_hour"])["predicted_demand"]
    .mean().reset_index()
)

best_times = best_times.sort_values(["region","predicted_demand"],ascending=[True,False]).groupby("region").head(3)
best_times["rank"] = best_times.groupby("region").cumcount()+1

region_mean = predictions.groupby("region")["predicted_demand"].mean()
predictions["best_region"] = region_mean.idxmax()
predictions["relocation_gain"] = region_mean.max() - predictions["predicted_demand"]

print("✅ Intelligence layer built")

✅ Intelligence layer built


In [27]:
# ================= BUSINESS INTELLIGENCE =================

# ---- 1. SURGE DETECTION ----
predictions["baseline"] = predictions.groupby("region")["actual_demand"].transform("mean")
predictions["baseline"] = predictions["baseline"].fillna(predictions["baseline"].median())

predictions["surge_threshold"] = predictions["baseline"] * 1.3
predictions["surge_flag"] = predictions["predicted_demand"] > predictions["surge_threshold"]

ratio = predictions["predicted_demand"] / (predictions["surge_threshold"] + 1e-6)

predictions["surge_level"] = "none"
predictions.loc[(predictions["surge_flag"]) & (ratio <= 1.1), "surge_level"] = "low"
predictions.loc[(predictions["surge_flag"]) & (ratio > 1.1) & (ratio <= 1.25), "surge_level"] = "medium"
predictions.loc[(predictions["surge_flag"]) & (ratio > 1.25), "surge_level"] = "high"

# ---- 2. RISK SCORE ----
predictions["std"] = predictions.groupby("region")["actual_demand"].transform("std").fillna(0)

predictions["risk_score"] = predictions["std"] / (predictions["baseline"] + 1e-6)

predictions["risk_band"] = pd.cut(
    predictions["risk_score"],
    bins=[-np.inf, 0.35, 0.75, np.inf],
    labels=["Stable", "Moderate", "Volatile"]
)

# ---- 3. REVENUE ----
if "avg_fare_region_slot" in predictions.columns:
    predictions["avg_fare"] = predictions["avg_fare_region_slot"]
elif "avg_fare_region_slot" in train_df.columns:
    predictions["avg_fare"] = train_df["avg_fare_region_slot"].median()
else:
    predictions["avg_fare"] = 10

predictions["expected_revenue"] = predictions["predicted_demand"] * predictions["avg_fare"]

# ---- 4. DEMAND PRESSURE ----
predictions["demand_pressure"] = predictions["predicted_demand"] / (predictions["baseline"] + 1e-6)

print("✅ Business intelligence ready")
predictions.head()

✅ Business intelligence ready


,actual_demand,predicted_demand,pickup_slot,region,rolling_mean,surge_threshold,surge_flag,surge_level,rolling_std,risk_score,risk_band,avg_fare,expected_revenue,demand_pressure,pickup_hour,pickup_day,best_region,relocation_gain,baseline,std
0,260.0,293.102966,2016-03-16 23:45:00,5,221.893824,288.461971,True,low,128.403003,0.578669,Moderate,14.499593,4249.873572,1.320915,23,2,3,-45.379059,221.893824,128.403003
1,219.0,218.520187,2016-03-16 23:45:00,23,156.969466,204.060305,True,low,82.860889,0.527879,Moderate,14.499593,3168.453670,1.392119,23,2,3,29.203720,156.969466,82.860889
2,206.0,272.005585,2016-03-16 23:45:00,24,226.553088,294.519015,False,none,118.463608,0.522896,Moderate,14.499593,3943.970136,1.200626,23,2,3,-24.281677,226.553088,118.463608
3,49.0,48.698505,2016-03-16 23:45:00,6,49.627342,64.515545,False,none,22.003991,0.443384,Moderate,14.499593,706.108484,0.981284,23,2,3,199.025406,49.627342,22.003991
4,187.0,251.076889,2016-03-16 23:45:00,26,141.720333,184.236433,True,high,83.666875,0.590366,Moderate,14.499593,3640.512577,1.771636,23,2,3,-3.352982,141.720333,83.666875


In [28]:
# ================= DRIVER RELOCATION =================

recommendations = []

for _, row in predictions.iterrows():

    current_region = row["region"]
    current_time = row["pickup_slot"]
    current_demand = row["predicted_demand"]

    same_time = predictions[predictions["pickup_slot"] == current_time]

    better = same_time[same_time["predicted_demand"] > current_demand]

    if len(better) == 0:
        continue

    best_target = better.sort_values("predicted_demand", ascending=False).iloc[0]

    recommendations.append({
        "region": current_region,
        "pickup_slot": current_time,
        "recommended_region": best_target["region"],
        "expected_gain": best_target["predicted_demand"] - current_demand
    })

recommendations_df = pd.DataFrame(recommendations)

print("✅ Relocation ready")
recommendations_df.head()

✅ Relocation ready


,region,pickup_slot,recommended_region,expected_gain
0,5,2016-03-16 23:45:00,17,57.650574
1,23,2016-03-16 23:45:00,17,132.233353
2,24,2016-03-16 23:45:00,17,78.747955
3,6,2016-03-16 23:45:00,17,302.055023
4,26,2016-03-16 23:45:00,17,99.676651


In [29]:
# ================= BEST TIME =================

if np.issubdtype(predictions["pickup_slot"].dtype, np.datetime64):

    predictions["hour"] = predictions["pickup_slot"].dt.hour

    slot_profile = predictions.groupby(
        ["region", "hour"]
    )["predicted_demand"].mean().reset_index()

    best_times = (
        slot_profile.sort_values(
            ["region", "predicted_demand"],
            ascending=[True, False]
        )
        .groupby("region")
        .head(3)
    )

    best_times["rank"] = best_times.groupby("region").cumcount() + 1

    print("✅ Best time ready")
    best_times.head()

✅ Best time ready


In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def safe_mape(y_true, y_pred):
    y_true = np.maximum(y_true, 10)
    return np.mean(np.abs((y_true - y_pred) / y_true))

def smape(y_true, y_pred):
    denom = np.abs(y_true) + np.abs(y_pred)
    return np.mean(2*np.abs(y_true-y_pred)/np.where(denom==0,1,denom))

def evaluate_metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE": float(safe_mape(y_true, y_pred)),
        "sMAPE": float(smape(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }

train_metrics = evaluate_metrics(y_train, y_pred_train)
test_metrics = evaluate_metrics(y_test, y_pred_test)

metrics_summary = pd.DataFrame([
    {"split":"train",**train_metrics},
    {"split":"test",**test_metrics},
])

print(metrics_summary)

   split        MAE       RMSE      MAPE     sMAPE        R2
0  train  12.122744  18.800429  0.170381  0.158406  0.975299
1   test  12.847234  20.211875  0.166926  0.165440  0.970114


In [31]:
from pathlib import Path
import joblib

base_path = Path("/kaggle/working/final_3")
base_path.mkdir(parents=True, exist_ok=True)

joblib.dump(model, base_path / "xgb_model.pkl")
metrics_summary.to_csv(base_path / "metrics_summary.csv", index=False)
predictions.to_csv(base_path / "predictions.csv", index=False)

ui_cols = [
    "pickup_slot","region","actual_demand","predicted_demand",
    "surge_flag","surge_level","risk_score","risk_band",
    "expected_revenue","demand_pressure",
]

predictions[[c for c in ui_cols if c in predictions.columns]].to_csv(
    base_path / "business_output.csv", index=False
)

if "recommendations_df" in globals() and not recommendations_df.empty:
    recommendations_df.to_csv(base_path / "driver_recommendations.csv", index=False)
else:
    pd.DataFrame().to_csv(base_path / "driver_recommendations.csv", index=False)

if "best_times" in globals() and not best_times.empty:
    best_times.to_csv(base_path / "best_times.csv", index=False)
else:
    pd.DataFrame().to_csv(base_path / "best_times.csv", index=False)

print("✅ All artifacts saved:", base_path)

✅ All artifacts saved: /kaggle/working/final_3


In [32]:
import shutil

src = "/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/final_data.csv"
dst = "/kaggle/working/final_data.csv"

shutil.copy(src, dst)

print("✅ Copied to working directory")

✅ Copied to working directory
